In [5]:
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans


# ========= 直接在这里改参数 =========
input_csv = "p0.8_mu0.2_n500_l5_1.csv"
#input_csv="synthetic_temporal_network.csv"
output_csv = "test_n500_l5.csv"

k = 10
n_clusters = 10
time_window = 300
step_decay = 1.0
random_state = 42
undirected = True
# ==================================


def build_temporal_adjacency(df, undirected=True):
    adj = defaultdict(list)

    for row in df.itertuples(index=False):
        s = row.source
        d = row.destination
        t = row.timestamp

        adj[s].append((d, t))
        if undirected:
            adj[d].append((s, t))

    for u in adj:
        adj[u].sort(key=lambda x: x[1])

    return adj


def temporal_transition_probs(neighbors, current_time, time_window):
    """
    给定某个节点的所有时序邻边 neighbors = [(v, t_edge), ...]
    只考虑窗口内的边:
        |t_edge - current_time| <= time_window

    对每个邻居 v，统计该窗口内与当前节点的连边数量 count_v，
    然后按 count_v 归一化作为转移概率。

    返回:
        next_probs = [(v, next_time, prob), ...]
    其中 next_time 取该邻居窗口内距离 current_time 最近的事件时间。
    """
    if not neighbors:
        return []

    neighbor_count = defaultdict(int)
    neighbor_best_time = {}
    neighbor_best_dt = {}

    for v, t_edge in neighbors:
        dt = abs(t_edge - current_time)
        if dt <= time_window:
            neighbor_count[v] += 1

            if (v not in neighbor_best_dt) or (dt < neighbor_best_dt[v]):
                neighbor_best_dt[v] = dt
                neighbor_best_time[v] = t_edge

    if not neighbor_count:
        return []

    total_count = sum(neighbor_count.values())
    if total_count <= 0:
        return []

    next_probs = []
    for v, cnt in neighbor_count.items():
        prob = cnt / total_count
        next_time = neighbor_best_time[v]
        next_probs.append((v, next_time, float(prob)))

    return next_probs


def temporal_k_hop_visit_distribution(start_node, start_time, adj, k, time_window, step_decay=1.0):
    """
    返回前 1..k 步累计访问节点分布，而不是只返回第 k 步终点分布。
    """
    state_dist = {(start_node, start_time): 1.0}
    visit_dist = defaultdict(float)

    for step in range(1, k + 1):
        new_state_dist = defaultdict(float)

        for (u, cur_t), p_state in state_dist.items():
            neighbors = adj.get(u, [])
            trans = temporal_transition_probs(
                neighbors=neighbors,
                current_time=cur_t,
                time_window=time_window
            )

            if not trans:
                new_state_dist[(u, cur_t)] += p_state
                continue

            for v, next_t, p_trans in trans:
                new_state_dist[(v, next_t)] += p_state * p_trans

        state_dist = dict(new_state_dist)

        step_weight = step_decay ** (step - 1)
        for (u, _t), p in state_dist.items():
            visit_dist[u] += step_weight * p

    total = sum(visit_dist.values())
    if total > 0:
        for u in visit_dist:
            visit_dist[u] /= total

    return dict(visit_dist)


def build_node_event_features(df, k, time_window, step_decay, undirected=True):
    all_nodes = pd.Index(
        pd.concat([df["source"], df["destination"]], ignore_index=True).unique()
    )
    node2id = {node: i for i, node in enumerate(all_nodes)}

    adj = build_temporal_adjacency(df, undirected=undirected)

    num_events = len(df)
    num_nodes = len(all_nodes)
    X = np.zeros((2 * num_events, num_nodes), dtype=np.float32)
    meta = []

    for row_idx, row in df.iterrows():
        s = row["source"]
        d = row["destination"]
        t = row["timestamp"]

        s_dist = temporal_k_hop_visit_distribution(
            start_node=s,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in s_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "source"))

        d_dist = temporal_k_hop_visit_distribution(
            start_node=d,
            start_time=t,
            adj=adj,
            k=k,
            time_window=time_window,
            step_decay=step_decay
        )
        feature_row = len(meta)
        for node, prob in d_dist.items():
            X[feature_row, node2id[node]] = prob
        meta.append((row_idx, "destination"))

    return X, meta, all_nodes


def run():
    df = pd.read_csv(input_csv)

    required_cols = {"source", "destination", "timestamp"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Input CSV 缺少必要列: {missing}")

    df = df.copy()
    df["orig_index"] = np.arange(len(df))
    df = df.sort_values(["timestamp", "orig_index"]).reset_index(drop=True)

    X, meta, nodes = build_node_event_features(
        df=df,
        k=k,
        time_window=time_window,
        step_decay=step_decay,
        undirected=undirected
    )

    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(X)

    source_commu = np.full(len(df), -1, dtype=int)
    destination_commu = np.full(len(df), -1, dtype=int)

    for label, (row_idx, role) in zip(labels, meta):
        if role == "source":
            source_commu[row_idx] = int(label)
        else:
            destination_commu[row_idx] = int(label)

    df["source_commu"] = source_commu
    df["destination_commu"] = destination_commu

    df = df.sort_values("orig_index").reset_index(drop=True)

    out_df = df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()
    out_df.to_csv(output_csv, index=False)

    print(f"Done. 输出保存到: {output_csv}")
    print(f"事件数: {len(df)}")
    print(f"节点数: {len(nodes)}")
    print(f"node-event数: {len(meta)}")
    print(
        f"k = {k}, time_window = {time_window}, "
        f"step_decay = {step_decay}, n_clusters = {n_clusters}"
    )


if __name__ == "__main__":
    run()

Done. 输出保存到: test_n500_l5.csv
事件数: 6263
节点数: 500
node-event数: 12526
k = 10, time_window = 300, step_decay = 1.0, n_clusters = 10
